# SISTEMA RAG COMPLETO  
### Orquestado mediante LangGraph

Realizado por 
- Juan Estevan Sinitave Rojas
- Juan Felipe Miranda Arciniegas
---

In [ ]:
# Instalar dependencias necesarias para el sistema RAG:
%pip install langchain-groq langchain-google-genai python-dotenv faiss-cpu sentence-transformers langchain-huggingface langchain-community langgraph -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# PARTE 0: IMPORTS Y CONFIGURACIÓN INICIAL
# ══════════════════════════════════════════════════════════════════════════════

import os
import json
import re
from pathlib import Path
from typing import TypedDict, Literal
from dotenv import load_dotenv

# LangChain + LangGraph
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, START, END

# Cargar variables de entorno
load_dotenv(Path("env/.env"))
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("Imports completados")

c:\Users\User\Documents\proyecto_RAG\env\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\User\Documents\proyecto_RAG\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports completados


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# PARTE 1: CONFIGURACIÓN DEL SISTEMA
# ══════════════════════════════════════════════════════════════════════════════

# Modelos LLM
GROQ_MODEL = "llama-3.3-70b-versatile"
GEMINI_MODEL = "models/gemini-2.5-flash-lite"

# Mapeo de intención a si requiere RAG
REQUIERE_RAG = {
    "busqueda"   : True,
    "resumen"    : True,
    "comparacion": True,
    "general"    : False,
}

# Top-K por defecto según intención
TOPK_DEFAULT = {
    "busqueda"   : 4,
    "resumen"    : 10,
    "comparacion": 12,
    "general"    : 0,
}

# Prompts del sistema por intención
SYSTEM_PROMPTS = {
    "busqueda": """Eres un experto en Biología Celular y Molecular.
Responde la pregunta ÚNICAMENTE usando los fragmentos proporcionados.
Cita con [1], [2], etc. Si no está en los fragmentos, di: 'No encontré esa información en el corpus.'""",

    "resumen": """Eres un experto en Biología Celular y Molecular.
Elabora un resumen estructurado ÚNICAMENTE usando los fragmentos proporcionados.
Orgániza con secciones claras. Cita con [1], [2], etc.""",

    "comparacion": """Eres un experto en Biología Celular y Molecular.
Compara los conceptos ÚNICAMENTE usando los fragmentos proporcionados.
Estructura: semejanzas, diferencias, conclusión. Cita con [1], [2], etc.""",
}

print(" Configuración del sistema lista")

 Configuración del sistema lista


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# PARTE 2: INICIALIZACIÓN DE VECTORSTORE Y LLMs
# ══════════════════════════════════════════════════════════════════════════════

# Cargar embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Cargar índice FAISS
vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True,
)

# Inicializar LLMs
def get_groq_llm() -> ChatGroq:
    """Retorna instancia de Groq (clasificación, evaluation)"""
    return ChatGroq(
        model=GROQ_MODEL,
        groq_api_key=GROQ_API_KEY,
        temperature=0,
    )

def get_gemini_llm() -> ChatGoogleGenerativeAI:
    """Retorna instancia de Gemini (generación de respuestas)"""
    return ChatGoogleGenerativeAI(
        model=GEMINI_MODEL,
        google_api_key=GOOGLE_API_KEY,
        temperature=0.2,
    )

# Instancias globales
llm_groq = get_groq_llm()
llm_gemini = get_gemini_llm()

print(f"Vectorstore cargado: {vectorstore}")
print(f"Groq inicializado: {GROQ_MODEL}")
print(f"Gemini inicializado: {GEMINI_MODEL}")

Vectorstore cargado: <langchain_community.vectorstores.faiss.FAISS object at 0x00000199D8ABE3C0>
Groq inicializado: llama-3.3-70b-versatile
Gemini inicializado: models/gemini-2.5-flash-lite


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PARTE 3: FUNCIONES AUXILIARES (Clasificación, Generación, Evaluación)
# ══════════════════════════════════════════════════════════════════════════════
# Esta sección contiene las funciones core del sistema:
# 1. CLASIFICACIÓN: Determina el tipo de consulta (búsqueda, resumen, comparación, general)
#    y selecciona dinámicamente cuántos documentos recuperar (top-k)
# 2. GENERACIÓN: Formatea el contexto, construye prompts y genera respuestas con LLM
# 3. EVALUACIÓN: Valida la calidad de respuestas en 3 dimensiones (respaldo, coherencia, completitud)

# ── CLASIFICACIÓN ──────────────────────────────────────────────────────────────
def clasificar_intencion(query: str, llm: ChatGroq = None) -> dict:
    """Clasifica intención y retorna dict con intent, confidence, reasoning, requiere_rag"""
    if llm is None:
        llm = get_groq_llm()
    
    system_prompt = """Eres un clasificador de intenciones para un sistema RAG de Biología.
Clasifica en: 'busqueda', 'resumen', 'comparacion', 'general'.
Responde ÚNICAMENTE con JSON: {"intent": "...", "confidence": 0.0-1.0, "reasoning": "..."}"""
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Consulta: {query}"),
    ]
    
    respuesta = llm.invoke(messages)
    texto = respuesta.content.strip()
    
    try:
        resultado = json.loads(texto)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", texto, re.DOTALL)
        resultado = json.loads(match.group()) if match else {
            "intent": "busqueda",
            "confidence": 0.5,
            "reasoning": "Fallback"
        }
    
    resultado["requiere_rag"] = REQUIERE_RAG.get(resultado["intent"], True)
    return resultado

def seleccionar_topk(query: str, intent: str, llm: ChatGroq = None) -> int:
    """Selecciona dinámicamente K según complejidad de consulta"""
    if llm is None:
        llm = get_groq_llm()
    
    if intent == "general":
        return 0
    
    rangos = {
        "busqueda": "entre 4 y 6",
        "resumen": "entre 8 y 12",
        "comparacion": "entre 10 y 14",
    }
    rango = rangos.get(intent, "entre 4 y 8")
    
    system_prompt = f"""Decide cuántos fragmentos recuperar.
Intención: {intent}, Rango: {rango}.
Responde JSON: {{"k": <número>, "reasoning": "..."}}"""
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Consulta: {query}"),
    ]
    
    respuesta = llm.invoke(messages)
    try:
        resultado = json.loads(respuesta.content.strip())
        k = int(resultado["k"])
    except:
        k = TOPK_DEFAULT.get(intent, 4)
    
    return max(1, min(k, 20))  # Clamp entre 1 y 20

# ── GENERACIÓN ────────────────────────────────────────────────────────────────
# Las funciones de generación convierten documentos FAISS en contexto formateado,
# construyen prompts personalizados según la intención, e invocan Gemini para
# generar respuestas que citen fragmentos originales

def formatear_contexto_y_citas(chunks: list[Document]) -> tuple[str, list[dict]]:
    """Convierte chunks en contexto numerado y lista de citas"""
    bloques = []
    citas = []
    
    for i, chunk in enumerate(chunks, start=1):
        meta = chunk.metadata
        bloques.append(f"[{i}] {chunk.page_content}")
        citas.append({
            "numero": i,
            "doc_id": meta.get("doc_id", "desconocido"),
            "page": meta.get("page", 0),
            "chunk_id": meta.get("chunk_id", f"chunk_{i}"),
            "preview": chunk.page_content[:80] + "...",
        })
    
    contexto_str = "\n\n".join(bloques)
    return contexto_str, citas

def construir_prompt(query: str, contexto_str: str, intent: str) -> tuple[str, str]:
    """Construye sistema + usuario prompts según intención"""
    system_prompt = SYSTEM_PROMPTS.get(intent, SYSTEM_PROMPTS["busqueda"])
    
    user_prompt = f"""FRAGMENTOS DEL CORPUS:
──────────────────────────────────────────
{contexto_str}
──────────────────────────────────────────

PREGUNTA: {query}

RESPUESTA:"""
    
    return system_prompt, user_prompt

def generar_respuesta(query: str, chunks: list[Document], intent: str = "busqueda",
                      llm: ChatGoogleGenerativeAI = None) -> dict:
    """Genera respuesta con contexto"""
    if llm is None:
        llm = get_gemini_llm()
    
    contexto_str, citas = formatear_contexto_y_citas(chunks)
    system_prompt, user_prompt = construir_prompt(query, contexto_str, intent)
    
    prompt_final = f"[SYSTEM]\n{system_prompt}\n\n[USER]\n{user_prompt}"
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ]
    
    respuesta_llm = llm.invoke(messages)
    respuesta = respuesta_llm.content.strip()
    
    return {
        "respuesta": respuesta,
        "prompt_final": prompt_final,
        "citas": citas,
        "intent": intent,
        "n_chunks": len(chunks),
    }

# ── EVALUACIÓN ────────────────────────────────────────────────────────────────
# La evaluación criterio es crucial para asegurar calidad: valida que la respuesta
# se base en los documentos (respaldo), sea lógicamente consistente (coherencia)
# y responda completamente la pregunta según su tipo (completitud)

def evaluar_respuesta(query: str, respuesta: str, chunks: list[Document], intent: str,
                      llm_groq: ChatGroq = None) -> dict:
    """Evalúa respuesta en 3 criterios: respaldo, coherencia, completitud"""
    if llm_groq is None:
        llm_groq = get_groq_llm()
    
    contexto_resumido = "\n".join([c.page_content[:150] for c in chunks[:3]])
    
    system_prompt = f"""Evalúa la respuesta en 3 dimensiones:
1. RESPALDO: ¿Se basa en los fragmentos?
2. COHERENCIA: ¿Es lógica y clara?
3. COMPLETITUD: ¿Responde completamente (intent={intent})?

FRAGMENTOS:
{contexto_resumido}

Responde JSON: {{"respaldo": true/false, "coherencia": true/false, "completitud": true/false, "confianza": 0.0-1.0, "retroalimentacion": "..."}}"""
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"PREGUNTA: {query}\n\nRESPUESTA: {respuesta}"),
    ]
    
    respuesta_eval = llm_groq.invoke(messages)
    texto = respuesta_eval.content.strip()
    
    try:
        resultado = json.loads(texto)
    except:
        match = re.search(r"\{.*\}", texto, re.DOTALL)
        resultado = json.loads(match.group()) if match else {
            "respaldo": True, "coherencia": True, "completitud": True,
            "confianza": 0.5, "retroalimentacion": "Fallback"
        }
    
    resultado["es_valida"] = (
        resultado.get("respaldo", True) and
        resultado.get("coherencia", True) and
        resultado.get("completitud", True)
    )
    
    return resultado

print("Funciones auxiliares definidas")

Funciones auxiliares definidas


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PARTE 4: LANGGRAPH - ORQUESTADOR CENTRAL CON TOOLS INTEGRADAS
# ══════════════════════════════════════════════════════════════════════════════
# 
# - Cada NODO representa una operación (clasificación, recuperación, generación, etc.)
# - Las ARISTAS definen transiciones entre nodos
# - Los DECISORES (conditional edges) elegem el camino según condiciones
# - Las TOOLS son funciones especializadas invocadas automáticamente según palabras clave
# 
# Flujo general:
#   Consulta → Clasificar intención → Selector (¿Tool/General/RAG?) → Procesar → Evaluar → Fin

# ── DEFINIR ESTADO DEL GRAFO ──────────────────────────────────────────────────
class RAGState(TypedDict):
    """Estado compartido a través del grafo
    
    Campos:
    - query: La pregunta del usuario
    - intent: Tipo de consulta (busqueda, resumen, comparacion, general)
    - tool_name: Nombre de la herramienta a ejecutar (si aplica)
    - top_k: Número de fragmentos a recuperar
    - chunks: Documentos recuperados del vectorstore
    - respuesta: Respuesta generada
    - evaluacion: Resultados de evaluación de calidad
    - intentos: Contador de reintentos
    - max_intentos: Máximo de reintentos permitidos
    - resultado_final: Resultado compilado al final del flujo
    - trazabilidad: Info para auditoría (prompts, citas usadas)
    """
    query: str
    intent: str
    tool_name: str  # Tool a ejecutar (si aplica)
    top_k: int
    chunks: list
    respuesta: str
    evaluacion: dict
    intentos: int
    max_intentos: int
    resultado_final: dict
    trazabilidad: dict

# ── FUNCIONES INTERNAS DE TOOLS ────────────────────────────────────────────────
# Las TOOLS son funciones especializadas que se invocan automáticamente cuando
# el usuario formula preguntas específicas. Cada tool genera un tipo de contenido
# diferente (flashcards, quizzes, comparaciones, etc.)

def tool_search_corpus(query: str, top_k: int = 5) -> list[dict]:
    """Tool interno: Búsqueda semántica en corpus
    
    Recupera los top-k fragmentos más similares a la consulta usando embeddings.
    Retorna lista de resultados con contenido, documento ID y página.
    """
    top_k = max(1, min(top_k, 20))
    chunks = vectorstore.similarity_search(query, k=top_k)
    
    resultados = []
    for i, chunk in enumerate(chunks, 1):
        meta = chunk.metadata
        resultados.append({
            "numero": i,
            "contenido": chunk.page_content[:150],
            "doc_id": meta.get("doc_id", "desconocido"),
            "page": meta.get("page", 0),
        })
    return resultados

def tool_generate_flashcards(tema: str, num_flashcards: int = 5) -> dict:
    """Tool interno: Generar flashcards educativas
    
    Crea tarjetas de estudio con preguntas y respuestas sobre un tema.
    Recupera contexto FAISS relevante y lo usa para generar preguntas significativas.
    """
    num_flashcards = max(1, min(num_flashcards, 10))
    chunks = vectorstore.similarity_search(tema, k=8)
    contexto = "\n".join([c.page_content[:200] for c in chunks])
    
    prompt = f"""Genera {num_flashcards} flashcards (Pregunta → Respuesta) sobre: "{tema}"
CONTEXTO: {contexto}
Responde SOLO con JSON: {{"flashcards": [{{"pregunta": "...", "respuesta": "...", "fuente": "..."}}]}}"""
    
    messages = [
        SystemMessage(content="Eres un generador de materiales educativos."),
        HumanMessage(content=prompt),
    ]
    respuesta = llm_groq.invoke(messages)
    
    try:
        resultado = json.loads(respuesta.content)
    except:
        match = re.search(r"\{.*\}", respuesta.content, re.DOTALL)
        resultado = json.loads(match.group()) if match else {"flashcards": []}
    
    return {"tema": tema, "cantidad": num_flashcards, **resultado}

def tool_generate_quiz(tema: str, num_preguntas: int = 3) -> dict:
    """Tool interno: Generar quizzes
    
    Crea preguntas de opción múltiple para evaluate el conocimiento.
    Incluye justificaciones para retroalimentación educativa.
    """
    num_preguntas = max(1, min(num_preguntas, 5))
    chunks = vectorstore.similarity_search(tema, k=10)
    contexto = "\n".join([c.page_content[:200] for c in chunks])
    
    prompt = f"""Genera {num_preguntas} preguntas de múltiple opción sobre: "{tema}"
CONTEXTO: {contexto}
Responde SOLO con JSON: {{"quiz": [{{"pregunta": "...", "opciones": [...], "respuesta_correcta": "...", "justificacion": "..."}}]}}"""
    
    messages = [
        SystemMessage(content="Eres un creador de quizzes educativos."),
        HumanMessage(content=prompt),
    ]
    respuesta = llm_groq.invoke(messages)
    
    try:
        resultado = json.loads(respuesta.content)
    except:
        match = re.search(r"\{.*\}", respuesta.content, re.DOTALL)
        resultado = json.loads(match.group()) if match else {"quiz": []}
    
    return {"tema": tema, "cantidad": num_preguntas, **resultado}

def tool_compare_concepts(concepto1: str, concepto2: str) -> dict:
    """Tool interno: Comparar conceptos
    
    Estructura comparación en semejanzas y diferencias.
    Recupera documentos para ambos conceptos y los compara sistemáticamente.
    """
    chunks1 = vectorstore.similarity_search(concepto1, k=5)
    chunks2 = vectorstore.similarity_search(concepto2, k=5)
    contexto = "\n".join([c.page_content[:150] for c in chunks1 + chunks2])
    
    prompt = f"""Compara "{concepto1}" vs "{concepto2}"
CONTEXTO: {contexto}
Responde SOLO con JSON: {{"comparacion": {{"semejanzas": [...], "diferencias": [...]}}}}"""
    
    messages = [
        SystemMessage(content="Eres un experto en comparación de conceptos biológicos."),
        HumanMessage(content=prompt),
    ]
    respuesta = llm_groq.invoke(messages)
    
    try:
        resultado = json.loads(respuesta.content)
    except:
        match = re.search(r"\{.*\}", respuesta.content, re.DOTALL)
        resultado = json.loads(match.group()) if match else {"comparacion": {}}
    
    return {"concepto1": concepto1, "concepto2": concepto2, **resultado}

def tool_summarize_topic(tema: str, num_secciones: int = 4) -> dict:
    """Tool interno: Resumir tema
    
    Genera resumen estructurado con secciones temáticas (definición, proceso, función, aplicaciones).
    Incluye puntos clave para síntesis rápida.
    """
    num_secciones = max(2, min(num_secciones, 5))
    chunks = vectorstore.similarity_search(tema, k=12)
    contexto = "\n".join([c.page_content[:200] for c in chunks])
    
    prompt = f"""Resume "{tema}" en {num_secciones} secciones: Definición, Proceso, Función, Aplicaciones
CONTEXTO: {contexto}
Responde SOLO con JSON: {{"secciones": [{{"titulo": "...", "contenido": "..."}}], "puntos_clave": [...]}}"""
    
    messages = [
        SystemMessage(content="Eres un experto en resúmenes educativos estructurados."),
        HumanMessage(content=prompt),
    ]
    respuesta = llm_groq.invoke(messages)
    
    try:
        resultado = json.loads(respuesta.content)
    except:
        match = re.search(r"\{.*\}", respuesta.content, re.DOTALL)
        resultado = json.loads(match.group()) if match else {"secciones": [], "puntos_clave": []}
    
    return {"tema": tema, "num_secciones": num_secciones, **resultado}
 
def tool_check_understanding(concepto: str, respuesta_estudiante: str) -> dict:
    """Tool interno: Validar comprensión del estudiante
    
    Evalúa si la respuesta del estudiante es correcta según el corpus.
    Proporciona puntuación, feedback constructivo y nivel de exactitud.
    """
    chunks = vectorstore.similarity_search(concepto, k=6)
    contexto = "\n".join([c.page_content[:200] for c in chunks])
    
    prompt = f"""Evalúa si esta respuesta de estudiante sobre "{concepto}" es correcta:
RESPUESTA: {respuesta_estudiante}
CONTEXTO CORRECTO: {contexto}
Responde SOLO con JSON: {{"es_valida": bool, "exactitud": "alta|media|baja", "feedback": "...", "puntuacion": 0-100}}"""
    
    messages = [
        SystemMessage(content="Eres un evaluador de respuestas de estudiantes en Biología."),
        HumanMessage(content=prompt),
    ]
    respuesta = llm_groq.invoke(messages)
    
    try:
        resultado = json.loads(respuesta.content)
    except:
        match = re.search(r"\{.*\}", respuesta.content, re.DOTALL)
        resultado = json.loads(match.group()) if match else {"es_valida": False, "exactitud": "baja"}
    
    return {"concepto": concepto, "validacion": resultado}

# ── MAPA DE TOOLS ──────────────────────────────────────────────────────────────
# Diccionario que mapea nombres de tools a sus funciones correspondientes
# Permite búsqueda dinámicamente y ejecución automática sin condicionales hardcoded
TOOLS_MAP = {
    "search_corpus": tool_search_corpus,
    "generate_flashcards": tool_generate_flashcards,
    "generate_quiz": tool_generate_quiz,
    "compare_concepts": tool_compare_concepts,
    "summarize_topic": tool_summarize_topic,
    "check_understanding": tool_check_understanding,
}

# ── NODOS DEL GRAFO ───────────────────────────────────────────────────────────
# Cada nodo es una función que procesa el estado actual y retorna estado actualizado

def node_classify(state: RAGState) -> RAGState:
    """Nodo 1: CLASIFICACIÓN
    
    Analiza la consulta para determinar:
    1. Intención (busqueda, resumen, comparacion, general)
    2. Top-k dinámico según complejidad
    3. Qué tool usar (si aplica), detectando palabras clave como:
       - flashcard, tarjeta → flashcards
       - quiz, examen → quiz
       - compara, vs → compare_concepts
       - resume → summarize_topic
       - verifica, entiende → check_understanding
       - busca → search_corpus
    """
    print(f"\n[NODO] Clasificación\n  Consulta: {state['query']}")
    
    clasificacion = clasificar_intencion(state["query"], llm_groq)
    state["intent"] = clasificacion["intent"]
    state["top_k"] = seleccionar_topk(state["query"], state["intent"], llm_groq)
    
    # Detectar qué tool se necesita basado en palabras clave
    query_lower = state["query"].lower()
    state["tool_name"] = ""  # Default: no tool
    
    if any(word in query_lower for word in ["flashcard", "tarjeta", "estudio", "memorizar"]):
        state["tool_name"] = "generate_flashcards"
    elif any(word in query_lower for word in ["quiz", "pregunta", "test", "examen"]):
        state["tool_name"] = "generate_quiz"
    elif any(word in query_lower for word in ["compara", "diferencia", "similitud", "vs"]):
        state["tool_name"] = "compare_concepts"
    elif any(word in query_lower for word in ["resume", "resumen", "síntesis"]):
        state["tool_name"] = "summarize_topic"
    elif any(word in query_lower for word in ["verifica", "entiende", "comprende", "correcto"]):
        state["tool_name"] = "check_understanding"
    elif any(word in query_lower for word in ["busca", "encuentra", "search"]):
        state["tool_name"] = "search_corpus"
    
    print(f"  → Intención: {state['intent']}")
    print(f"  → Top-K: {state['top_k']}")
    if state["tool_name"]:
        print(f"  → Tool detectada: {state['tool_name']}")
    
    return state

def decisor_herramientas(state: RAGState) -> Literal["tool", "general", "rag"]:
    """SELECTOR 1: ¿Usar tool especializada, respuesta general o flujo RAG?
    
    - "tool": Se invoca una herramienta específica
    - "general": Respuesta sin documentos (para preguntas fuera del dominio)
    - "rag": Flujo completo RAG (recuperar → generar → evaluar)
    """
    if state["tool_name"]:
        return "tool"
    elif state["intent"] == "general":
        return "general"
    else:
        return "rag"

def node_execute_tool(state: RAGState) -> RAGState:
    """Nodo 2a: EJECUCIÓN DE TOOL
    
    Ejecuta la herramienta detectada en nodo_classify.
    También recupera documentos FAISS para auditoría y trazabilidad,
    asegurando que TODAS las respuestas sean auditables.
    """
    print(f"\n[NODO] Ejecución de Tool: {state['tool_name']}")
    
    try:
        # TODAS las tools generan respuesta directamente
        tool_func = TOOLS_MAP.get(state["tool_name"])
        if tool_func:
            resultado = tool_func(state["query"], state["top_k"])
            state["respuesta"] = str(resultado)
            
            # IMPORTANTE: Capturar y trackear documentos FAISS para TODAS las tools
            # Esto permite auditoría y trazabilidad completa
            chunks_topic = vectorstore.similarity_search(state["query"], k=state["top_k"])
            state["chunks"] = chunks_topic
            
            # NUEVO: Convertir chunks a citas para trazabilidad (igual que en node_generate)
            _, citas = formatear_contexto_y_citas(chunks_topic)
            state["trazabilidad"] = {
                "citas": citas,
            }
            
            print(f"  → Tool ejecutada exitosamente")
            print(f"  → {len(chunks_topic)} documentos FAISS trackeados para auditoria")
            
    except Exception as e:
        print(f"  → Error en tool: {str(e)}")
        state["respuesta"] = f"Error ejecutando herramienta: {str(e)}"
        state["chunks"] = []
        state["trazabilidad"] = {"citas": []}
        state["evaluacion"] = {"es_valida": False, "error": str(e)}
    
    return state

def node_general(state: RAGState) -> RAGState:
    """Nodo 2b: RESPUESTA GENERAL (sin RAG)
    
    Para preguntas que no requieren documentos (e.g., "¿Cuántos continentes hay?").
    Invoca Gemini sin contexto de corpus.
    """
    print("\n[NODO] Respuesta General (sin RAG)")
    
    messages = [
        SystemMessage(content="Eres un experto en Biología Celular y Molecular. Responde de forma clara y precisa."),
        HumanMessage(content=state["query"]),
    ]
    
    respuesta_llm = llm_gemini.invoke(messages)
    state["respuesta"] = respuesta_llm.content.strip()
    state["chunks"] = []
    state["evaluacion"] = {"es_valida": True, "tipo": "general", "confianza": 1.0}
    
    print("  → Respuesta generada")
    return state

def node_retrieve(state: RAGState) -> RAGState:
    """Nodo 2c: RECUPERACIÓN
    
    Búsqueda semántica en el vectorstore FAISS.
    Recupera top_k fragmentos más similares a la consulta usando embeddings.
    """
    print(f"\n[NODO] Recuperación (Top-K: {state['top_k']})")
    
    chunks = vectorstore.similarity_search(state["query"], k=state["top_k"])
    state["chunks"] = chunks
    
    print(f"  → {len(chunks)} fragmentos recuperados")
    return state

def node_generate(state: RAGState) -> RAGState:
    """Nodo 3: GENERACIÓN
    
    Genera respuesta usando Gemini con contexto.
    Formatea fragmentos FAISS, construye prompt personalizado según intención,
    e invoca LLM con instrucciones de citas.
    """
    print("\n[NODO] Generación")
    
    resultado = generar_respuesta(state["query"], state["chunks"], state["intent"], llm_gemini)
    state["respuesta"] = resultado["respuesta"]
    state["trazabilidad"] = {
        "prompt_final": resultado["prompt_final"],
        "citas": resultado["citas"],
    }
    
    print("  → Respuesta generada con contexto")
    return state

def node_evaluate(state: RAGState) -> RAGState:
    """Nodo 4: EVALUACIÓN
    
    Valida calidad de respuesta en 3 dimensiones:
    1. RESPALDO: ¿Se basa en los fragmentos recuperados?
    2. COHERENCIA: ¿Es lógica y bien escrita?
    3. COMPLETITUD: ¿Responde completamente según el tipo de consulta?
    
    Usa estos criterios para decidir si aceptar o reparar.
    """
    print("\n[NODO] Evaluación")
    
    evaluacion = evaluar_respuesta(
        state["query"], state["respuesta"], state["chunks"], state["intent"], llm_groq
    )
    state["evaluacion"] = evaluacion
    
    if evaluacion["es_valida"]:
        print("  → VÁLIDA (paso evaluación)")
    else:
        print(f"  → INVÁLIDA ({evaluacion.get('retroalimentacion', 'sin detalles')})")
    
    return state

def decisor_evaluacion(state: RAGState) -> Literal["fin", "reparar"]:
    """SELECTOR 2: ¿Finalizar o continuar reparando?
    
    Si validación falla y hay intentos disponibles, reparar (aumentar top-k).
    Para no agotar cuota de API, SIEMPRE finaliza en esta versión.
    """
    # SIEMPRE finaliza sin reparaciones para no agotar cuota
    return "fin"

def node_repair(state: RAGState) -> RAGState:
    """Nodo 5: REPARACIÓN (opcional, no se invoca actualmente)
    
    Si respuesta falla evaluación, aumenta top_k y reintenta.
    Implementa estrategia adaptativa de recuperación.
    """
    print("\n[NODO] Reparación")
    
    old_k = state["top_k"]
    state["top_k"] = min(state["top_k"] + 2, 20)
    state["intentos"] += 1
    
    print(f"  → Intento {state['intentos']}: Top-K {old_k} → {state['top_k']}")
    return state

def node_finalize(state: RAGState) -> RAGState:
    """Nodo 6: FINALIZACIÓN
    
    Prepara resultado final compilando:
    - Consulta y intención
    - Respuesta generada
    - Estadísticas (chunks usados, confianza, intentos)
    - Información de trazabilidad (para auditoría)
    - Evaluación completa
    """
    print(f"\n[NODO] Finalización (Intentos: {state['intentos']})")
    
    estado_final = {
        "query": state["query"],
        "intent": state["intent"],
        "tool_usado": state.get("tool_name", "ninguno"),
        "respuesta": state["respuesta"],
        "chunks_usados": len(state["chunks"]),
        "evaluacion_valida": state["evaluacion"].get("es_valida", False),
        "confianza": state["evaluacion"].get("confianza", 0.0),
        "intentos": state["intentos"],
        "trazabilidad": state.get("trazabilidad", {}),
        "evaluacion": state["evaluacion"],
    }
    state["resultado_final"] = estado_final
    
    print("  → Resultado final preparado")
    return state

def decisor_post_tool(state: RAGState) -> Literal["generate", "evaluate"]:
    """SELECTOR 3: Después de tool - ¿Generar respuesta natural o evaluar directamente?
    
    - search_corpus retorna lista de documentos pero necesita respuesta elaborada → generate
    - Otras tools retornan JSON estructurado ya procesado → evaluate
    """
    if state["tool_name"] == "search_corpus" and state["chunks"]:
        return "generate"  # search_corpus necesita generar respuesta elaborada con documento
    else:
        return "evaluate"  # Otras tools retornan JSON ya procesado, evalúa y finaliza

# ── CONSTRUIR GRAFO ───────────────────────────────────────────────────────────
def construir_grafo() -> object:
    """Construye y retorna el grafo compilado con tools integradas
    
    Arquitectura del grafo:
    START → classify → [decisor_herramientas] →
       ├─ tool → [decisor_post_tool] → {generate|evaluate}
       ├─ general → finalize
       └─ rag → retrieve → generate → evaluate → [decisor_evaluacion] → {repair|finalize}
    → END
    """
    workflow = StateGraph(RAGState)
    
    # Agregar nodos
    workflow.add_node("classify", node_classify)
    workflow.add_node("tool", node_execute_tool)
    workflow.add_node("general", node_general)
    workflow.add_node("retrieve", node_retrieve)
    workflow.add_node("generate", node_generate)
    workflow.add_node("evaluate", node_evaluate)
    workflow.add_node("repair", node_repair)
    workflow.add_node("finalize", node_finalize)
    
    # Conectar aristas
    workflow.add_edge(START, "classify")
    
    # Router: ¿Tool, General o RAG?
    workflow.add_conditional_edges(
        "classify",
        decisor_herramientas,
        {"tool": "tool", "general": "general", "rag": "retrieve"}
    )
    
    # Ramas después de tool (decide si necesita generar respuesta natural o evaluar directamente)
    workflow.add_conditional_edges(
        "tool",
        decisor_post_tool,
        {"generate": "generate", "evaluate": "evaluate"}
    )
    
    # Ramas finales
    workflow.add_edge("general", "finalize")
    workflow.add_edge("retrieve", "generate")
    workflow.add_edge("generate", "evaluate")
    
    # Router: ¿Finalizar o reparar?
    workflow.add_conditional_edges(
        "evaluate",
        decisor_evaluacion,
        {"reparar": "repair", "fin": "finalize"}
    )
    
    workflow.add_edge("repair", "retrieve")  # Loop back
    workflow.add_edge("finalize", END)
    
    return workflow.compile()

# Compilar grafo globalmente
grafo_rag = construir_grafo()

print("LangGraph definido y compilado (con tools integradas)")

LangGraph definido y compilado (con tools integradas)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PARTE 5: FUNCIONES DE EJECUCIÓN Y VISUALIZACIÓN
# ══════════════════════════════════════════════════════════════════════════════
# Esta sección contiene las funciones para ejecutar el sistem RAG completo
# y visualizar resultados de forma legible y amigable.

def ejecutar_rag(query: str, max_intentos: int = 0) -> dict:
    """Ejecuta el grafo RAG completo y retorna resultado final
    
    Pasos:
    1. Crea estado inicial de la consulta
    2. Invoca el grafo (que ejecuta toda la orquestación)
    3. Retorna resultado final compilado
    
    Args:
        query: Pregunta del usuario
        max_intentos: Reintentos máximos si falla evaluación (no usado actualmente)
    
    Returns:
        dict con: pregunta, intención, respuesta, chunks, evaluación, trazabilidad
    """
    print("\n" + "=" * 80)
    print("EJECUCIÓN DEL SISTEMA RAG")
    print("=" * 80)
    
    initial_state = RAGState(
        query=query,
        intent="",
        top_k=0,
        chunks=[],
        respuesta="",
        evaluacion={},
        intentos=0,
        max_intentos=max_intentos,
        resultado_final={},
        trazabilidad={},
    )
    
    final_state = grafo_rag.invoke(initial_state)
    return final_state["resultado_final"]

def mostrar_resultado(resultado: dict):
    """Muestra resultado de forma legible y amigable + referencias RAG
    
    Formatea la respuesta según su tipo:
    - Flashcards: Muestra preguntas/respuestas formateadas
    - Quizzes: Muestra preguntas con opciones y justificaciones
    - Comparaciones: Muestra semejanzas y diferencias
    - Resúmenes: Muestra secciones organizadas
    - Búsquedas: Muestra respuesta natural con citas
    - General: Muestra texto plano
    
    SIEMPRE muestra referencias FAISS usadas (trazabilidad).
    """
    import ast
    
    linea = "=" * 80
    print("\n" + linea)
    print("RESULTADO FINAL")
    print(linea)
    print(f"Intención: {resultado['intent']}")
    print(f"Tool usada: {resultado['tool_usado']}")
    print(f"Chunks usados: {resultado['chunks_usados']}")
    print(f"Válida: {'SÍ' if resultado['evaluacion_valida'] else 'NO'}")
    print(f"Confianza: {resultado['confianza']:.0%}")
    
    linea2 = "─" * 80
    print("\n" + linea2)
    print("RESPUESTA:")
    print(linea2)
    
    respuesta_str = resultado["respuesta"]
    
    # Intentar parsear como diccionario si parece serlo
    try:
        if respuesta_str.startswith("{") and respuesta_str.endswith("}"):
            respuesta_dict = ast.literal_eval(respuesta_str)
            
            # Formatear según tipo de tool
            if resultado['tool_usado'] == 'generate_flashcards':
                print(f"TEMA: {respuesta_dict.get('tema', 'N/A')}")
                print(f"CANTIDAD: {respuesta_dict.get('cantidad', 'N/A')} flashcards\n")
                for i, fc in enumerate(respuesta_dict.get('flashcards', []), 1):
                    print(f"  Flashcard {i}:")
                    print(f"    Pregunta: {fc.get('pregunta', 'N/A')}")
                    print(f"    Respuesta: {fc.get('respuesta', 'N/A')}")
                    print(f"    Fuente: {fc.get('fuente', 'N/A')}\n")
            
            elif resultado['tool_usado'] == 'generate_quiz':
                print(f"TEMA: {respuesta_dict.get('tema', 'N/A')}")
                print(f"CANTIDAD: {respuesta_dict.get('cantidad', 'N/A')} preguntas\n")
                for i, q in enumerate(respuesta_dict.get('quiz', []), 1):
                    print(f"  Pregunta {i}: {q.get('pregunta', 'N/A')}")
                    opciones = q.get('opciones', [])
                    for j, op in enumerate(opciones, 1):
                        print(f"    {chr(96+j)}) {op}")
                    print(f"    Respuesta correcta: {q.get('respuesta_correcta', 'N/A')}")
                    print(f"    Justificación: {q.get('justificacion', 'N/A')}\n")
            
            elif resultado['tool_usado'] == 'compare_concepts':
                print(f"COMPARACIÓN: {respuesta_dict.get('concepto1', 'N/A')} vs {respuesta_dict.get('concepto2', 'N/A')}\n")
                comp = respuesta_dict.get('comparacion', {})
                print(f"  SEMEJANZAS:")
                for s in comp.get('semejanzas', []):
                    print(f"    • {s}")
                print(f"\n  DIFERENCIAS:")
                for d in comp.get('diferencias', []):
                    print(f"    • {d}\n")
            
            elif resultado['tool_usado'] == 'summarize_topic':
                print(f"TEMA: {respuesta_dict.get('tema', 'N/A')}")
                print(f"SECCIONES: {respuesta_dict.get('num_secciones', 'N/A')}\n")
                for i, sec in enumerate(respuesta_dict.get('secciones', []), 1):
                    print(f"  {i}. {sec.get('titulo', 'N/A').upper()}")
                    print(f"     {sec.get('contenido', 'N/A')}\n")
                print(f"  PUNTOS CLAVE:")
                for pk in respuesta_dict.get('puntos_clave', []):
                    print(f"    • {pk}\n")
            
            elif resultado['tool_usado'] == 'search_corpus':
                print("\n".join([f"  [{i}] {r.get('contenido', 'N/A')}" for i, r in enumerate(respuesta_dict if isinstance(respuesta_dict, list) else respuesta_dict.get('resultados', []), 1)]))
            
            elif resultado['tool_usado'] == 'check_understanding':
                print(f"CONCEPTO: {respuesta_dict.get('concepto', 'N/A')}\n")
                val = respuesta_dict.get('validacion', {})
                print(f"  ¿VÁLIDA?: {'Sí' if val.get('es_valida') else 'No'}")
                print(f"  EXACTITUD: {val.get('exactitud', 'N/A')}")
                print(f"  PUNTUACIÓN: {val.get('puntuacion', 'N/A')}/100")
                print(f"  FEEDBACK: {val.get('feedback', 'N/A')}\n")
            else:
                # Print JSON formatted para otros casos
                print(json.dumps(respuesta_dict, indent=2, ensure_ascii=False))
        else:
            print(respuesta_str[:800] + ("..." if len(respuesta_str) > 800 else ""))
    except:
        # Si no es diccionario, mostrar como texto normal
        print(respuesta_str[:800] + ("..." if len(respuesta_str) > 800 else ""))
    
    # SIEMPRE mostrar referencias/citas del RAG (trackeo obligatorio)
    print("\n" + linea2)
    print("REFERENCIAS RAG UTILIZADAS:")
    print(linea2)
    
    if resultado["trazabilidad"].get("citas"):
        for cita in resultado["trazabilidad"]["citas"]:
            print(f"[{cita['numero']}] Documento: {cita['doc_id']}")
            print(f"    Chunk ID: {cita['chunk_id']}")
            print(f"    Página: {cita['page']}")
            print(f"    Preview: {cita['preview']}\n")
    else:
        print("Sin referencias (respuesta sin documentos)\n")
    
    print("\n" + linea + "\n")

def mostrar_estructura_grafo():
    """Muestra diagrama textual de la estructura de LangGraph
    
    Visualiza:
    - Nodos y sus funciones
    - Conectividad y flujos condicionales
    - Herramientas disponibles
    - Criterios de evaluación
    """
    print("\n" + "=" * 80)
    print("ESTRUCTURA DEL GRAFO RAG")
    print("=" * 80)
    print("""
FLUJO:
  START
    ↓
  [classify] → Clasificar intención y seleccionar top-K
    ↓
  [decisor_herramientas] → ¿Tool, General o RAG?
    ├─ [tool] → Ejecutar herramienta específica → [finalize]
    ├─ [general] → Responder sin documentos → [finalize]
    └─ [retrieve] → Recuperar fragmentos
         ↓
       [generate] → Generar respuesta con contexto
         ↓
       [evaluate] → Evaluar en 3 criterios
         ↓
       [ROUTER] → ¿Válida?
         ├─ [repair] → Aumentar top-K y reintentar → [retrieve] (loop)
         └─ [finalize] → Preparar resultado
    ↓
  END

HERRAMIENTAS DISPONIBLES (invocación automática):
  1. search_corpus - Búsqueda semántica en corpus
  2. generate_flashcards - Generar tarjetas estudiantiles
  3. generate_quiz - Crear quizzes interactivos
  4. compare_concepts - Comparación estructurada
  5. summarize_topic - Resumen con secciones
  6. check_understanding - Validación de respuestas

CRITERIOS DE EVALUACIÓN:
  • Respaldo: ¿Se basa en los fragmentos?
  • Coherencia: ¿Es lógica y clara?
  • Completitud: ¿Responde completamente?

SI FALLA: Aumenta top-K y reintenta automáticamente.
""")


print("Sistema RAG completamente definido y listo")

Sistema RAG completamente definido y listo


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DEMO 1: VISUALIZAR ESTRUCTURA DEL GRAFO
# ══════════════════════════════════════════════════════════════════════════════
# Esta celda muestra el diagrama textual de cómo funciona LangGraph:
# - Nodos disponibles (clasificación, recuperación, generación, evaluación)
# - Flujos condicionales (decisores)
# - Herramientas integradas
# - Criterios de calidad

mostrar_estructura_grafo()


ESTRUCTURA DEL GRAFO RAG

FLUJO:
  START
    ↓
  [classify] → Clasificar intención y seleccionar top-K
    ↓
  [decisor_herramientas] → ¿Tool, General o RAG?
    ├─ [tool] → Ejecutar herramienta específica → [finalize]
    ├─ [general] → Responder sin documentos → [finalize]
    └─ [retrieve] → Recuperar fragmentos
         ↓
       [generate] → Generar respuesta con contexto
         ↓
       [evaluate] → Evaluar en 3 criterios
         ↓
       [ROUTER] → ¿Válida?
         ├─ [repair] → Aumentar top-K y reintentar → [retrieve] (loop)
         └─ [finalize] → Preparar resultado
    ↓
  END

HERRAMIENTAS DISPONIBLES (invocación automática):
  1. search_corpus - Búsqueda semántica en corpus
  2. generate_flashcards - Generar tarjetas estudiantiles
  3. generate_quiz - Crear quizzes interactivos
  4. compare_concepts - Comparación estructurada
  5. summarize_topic - Resumen con secciones
  6. check_understanding - Validación de respuestas

CRITERIOS DE EVALUACIÓN:
  • Respaldo: ¿Se basa

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PRUEBA 1: CONSULTA DE BÚSQUEDA
# ══════════════════════════════════════════════════════════════════════════════
# Prueba el flujo RAG básico: clasificar intención como "búsqueda",
# recuperar documentos relevantes del corpus, generar respuesta con citas,
# y evaluar calidad. Se espera que cite los fragmentos del corpus.

resultado = ejecutar_rag("¿Que es el The endoplasmic reticulum (ER)?")
mostrar_resultado(resultado)


EJECUCIÓN DEL SISTEMA RAG

[NODO] Clasificación
  Consulta: ¿Que es el The endoplasmic reticulum (ER)?
  → Intención: busqueda
  → Top-K: 4

[NODO] Recuperación (Top-K: 4)
  → 4 fragmentos recuperados

[NODO] Generación
  → Respuesta generada con contexto

[NODO] Evaluación
  → VÁLIDA (paso evaluación)

[NODO] Finalización (Intentos: 0)
  → Resultado final preparado

RESULTADO FINAL
Intención: busqueda
Tool usada: 
Chunks usados: 4
Válida: SÍ
Confianza: 90%

────────────────────────────────────────────────────────────────────────────────
RESPUESTA:
────────────────────────────────────────────────────────────────────────────────
El retículo endoplasmático (RE) es el orgánulo principal responsable del plegamiento correcto de las proteínas [2]. Mantiene la proteostasis a través de una red coordinada de chaperonas y vías de degradación que rigen el plegamiento, control de calidad y secreción de proteínas [4]. Una vía clave en este sistema es la vía de degradación asociada al RE (ERAD), qu

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PRUEBA 2: OTRA CONSULTA DE BÚSQUEDA
# ══════════════════════════════════════════════════════════════════════════════
# Otra prueba del flujo RAG con términos biológicos diferentes.
# Verifica que el sistema puede manejar múltiples conceptos del dominio.

resultado = ejecutar_rag("¿Que es Chromosome segregation?")
mostrar_resultado(resultado)


EJECUCIÓN DEL SISTEMA RAG

[NODO] Clasificación
  Consulta: ¿Que es Chromosome segregation?
  → Intención: busqueda
  → Top-K: 4

[NODO] Recuperación (Top-K: 4)
  → 4 fragmentos recuperados

[NODO] Generación
  → Respuesta generada con contexto

[NODO] Evaluación
  → VÁLIDA (paso evaluación)

[NODO] Finalización (Intentos: 0)
  → Resultado final preparado

RESULTADO FINAL
Intención: busqueda
Tool usada: 
Chunks usados: 4
Válida: SÍ
Confianza: 90%

────────────────────────────────────────────────────────────────────────────────
RESPUESTA:
────────────────────────────────────────────────────────────────────────────────
Chromosome segregation is the essential biological processes in which chromosomes are partitioned into the two daughter cells during cell division [1].

────────────────────────────────────────────────────────────────────────────────
REFERENCIAS RAG UTILIZADAS:
────────────────────────────────────────────────────────────────────────────────
[1] Documento: Central-spindle_

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PRUEBA 3: CONSULTA DE RESUMEN
# ══════════════════════════════════════════════════════════════════════════════
# Prueba el flujo RAG con intención "resumen": clasificador debe reconocer
# que se pide resumir, recuperar más documentos (top-k mayor) para síntesis,
# generar resumen estructurado con múltiples aspectos del tema.

resultado = ejecutar_rag("Resume el articulo de Carlos Gutiérrez-Merino: Synergy of Theory and Experimentation in Biological Membrane Research")
mostrar_resultado(resultado)


EJECUCIÓN DEL SISTEMA RAG

[NODO] Clasificación
  Consulta: Resume el articulo de Carlos Gutiérrez-Merino: Synergy of Theory and Experimentation in Biological Membrane Research


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kavz1aaeec49mmrwbxkmg3cd` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 100000, Requested 107. Please try again in 1m32.448s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DEMO 3: TOOL GENERATE_FLASHCARDS - INVOCACIÓN AUTOMÁTICA
# ══════════════════════════════════════════════════════════════════════════════
# El nodo de clasificación detecta palabras clave "flashcards" en la consulta
# e invoca automáticamente la tool "generate_flashcards".
# 
# Flujo:
#   1. Consulta incluye palabra clave → "generate_flashcards" se detecta
#   2. Tool se ejecuta: recupera 8 documentos FAISS sobre el tema
#   3. Genera JSON con flashcards (pregunta → respuesta)
#   4. Se evalúa y finaliza (sin flujo RAG completo)
# 
# Resultado: Tarjetas de estudio basadas en corpus para memorización.

print("\n" + "="*80)
print("DEMO 3: Flashcards - Tool 'generate_flashcards' se invoca AUTOMÁTICAMENTE")
print("="*80)

resultado = ejecutar_rag("Genera flashcards sobre la vida de la membrana celular para estudiar")
mostrar_resultado(resultado)


DEMO 3: Flashcards - Tool 'generate_flashcards' se invoca AUTOMÁTICAMENTE

EJECUCIÓN DEL SISTEMA RAG

[NODO] Clasificación
  Consulta: Genera flashcards sobre la vida de la membrana celular para estudiar
  → Intención: busqueda
  → Top-K: 4
  → Tool detectada: generate_flashcards

[NODO] Ejecución de Tool: generate_flashcards
  → Tool ejecutada exitosamente
  → 4 documentos FAISS trackeados para auditoria

[NODO] Evaluación
  → INVÁLIDA (La respuesta proporciona información relevante y útil sobre la vida de la membrana celular, pero no se basa en los fragmentos proporcionados. La respuesta es clara y lógica, y responde completamente a la pregunta. Sin embargo, la confianza es moderada debido a que la respuesta no se basa en los fragmentos proporcionados.)

[NODO] Finalización (Intentos: 0)
  → Resultado final preparado

RESULTADO FINAL
Intención: busqueda
Tool usada: generate_flashcards
Chunks usados: 4
Válida: NO
Confianza: 80%

───────────────────────────────────────────────────────

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DEMO 4: TOOL GENERATE_QUIZ - INVOCACIÓN AUTOMÁTICA
# ══════════════════════════════════════════════════════════════════════════════
# El nodo de clasificación detecta palabras clave "quiz" en la consulta
# e invoca automáticamente la tool "generate_quiz".
# 
# Flujo:
#   1. Consulta incluye palabra clave "quiz" → tool se detecta
#   2. Tool se ejecuta: recupera 10 documentos FAISS sobre el tema
#   3. Genera JSON con preguntas de múltiple opción y justificaciones
#   4. Se evalúa y finaliza
# 
# Resultado: Evaluación interactiva con preguntas y retroalimentación.

print("\n" + "="*80)
print("DEMO 4: Quiz - Tool 'generate_quiz' se invoca AUTOMÁTICAMENTE")
print("="*80)

resultado = ejecutar_rag("Crea un quiz sobre Drosophila spermatocytes")
mostrar_resultado(resultado)


DEMO 4: Quiz - Tool 'generate_quiz' se invoca AUTOMÁTICAMENTE

EJECUCIÓN DEL SISTEMA RAG

[NODO] Clasificación
  Consulta: Crea un quiz sobre Drosophila spermatocytes
  → Intención: busqueda
  → Top-K: 4
  → Tool detectada: generate_quiz

[NODO] Ejecución de Tool: generate_quiz
  → Tool ejecutada exitosamente
  → 4 documentos FAISS trackeados para auditoria

[NODO] Evaluación
  → VÁLIDA (paso evaluación)

[NODO] Finalización (Intentos: 0)
  → Resultado final preparado

RESULTADO FINAL
Intención: busqueda
Tool usada: generate_quiz
Chunks usados: 4
Válida: SÍ
Confianza: 90%

────────────────────────────────────────────────────────────────────────────────
RESPUESTA:
────────────────────────────────────────────────────────────────────────────────
TEMA: Crea un quiz sobre Drosophila spermatocytes
CANTIDAD: 4 preguntas

  Pregunta 1: ¿Por qué Drosophila spermatocytes son un sistema adecuado para el estudio de la citocinesis?
    a) Debido a su pequeño tamaño
    b) Debido a la disponibilida

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DEMO 5: TOOL COMPARE_CONCEPTS - INVOCACIÓN AUTOMÁTICA
# ══════════════════════════════════════════════════════════════════════════════
# El nodo de clasificación detecta palabras clave "compara" o "vs" en la consulta
# e invoca automáticamente la tool "compare_concepts".
# 
# Flujo:
#   1. Consulta incluye "compara" o "vs" → tool se detecta
#   2. Tool se ejecuta: recupera documentos sobre AMBOS conceptos
#   3. Genera JSON structurado con semejanzas y diferencias
#   4. Se evalúa y finaliza
# 
# Resultado: Análisis comparativo estructura de dos conceptos biológicos.

print("\n" + "="*80)
print("DEMO 5: Comparación - Tool 'compare_concepts' se invoca AUTOMÁTICAMENTE")
print("="*80)

resultado = ejecutar_rag("Compara RNA vs MicroRNA")
mostrar_resultado(resultado)


DEMO 5: Comparación - Tool 'compare_concepts' se invoca AUTOMÁTICAMENTE

EJECUCIÓN DEL SISTEMA RAG

[NODO] Clasificación
  Consulta: Compara RNA vs MicroRNA
  → Intención: comparacion
  → Top-K: 12
  → Tool detectada: compare_concepts

[NODO] Ejecución de Tool: compare_concepts
  → Error en tool: 'int' object has no attribute 'replace'

[NODO] Evaluación


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kavz1aaeec49mmrwbxkmg3cd` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 100000, Requested 182. Please try again in 2m37.248s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DEMO 6: TOOL SUMMARIZE_TOPIC - INVOCACIÓN AUTOMÁTICA
# ══════════════════════════════════════════════════════════════════════════════
# El nodo de clasificación detecta palabras clave "resume" en la consulta
# e invoca automáticamente la tool "summarize_topic".
# 
# Flujo:
#   1. Consulta incluye "resume" → tool se detecta
#   2. Tool se ejecuta: recupera 12 documentos FAISS (máximo para síntesis)
#   3. Genera JSON con secciones (Definición, Proceso, Función, Aplicaciones)
#   4. Se evalúa y finaliza
# 
# Resultado: Resumen estructurado con puntos clave para estudio rápido.

print("\n" + "="*80)
print("DEMO 6: Resumen - Tool 'summarize_topic' se invoca AUTOMÁTICAMENTE")
print("="*80)

resultado = ejecutar_rag("Resume el tema de Nitrile hydratases")
mostrar_resultado(resultado)


DEMO 6: Resumen - Tool 'summarize_topic' se invoca AUTOMÁTICAMENTE

EJECUCIÓN DEL SISTEMA RAG

[NODO] Clasificación
  Consulta: Resume el tema de Nitrile hydratases
  → Intención: resumen
  → Top-K: 10
  → Tool detectada: summarize_topic

[NODO] Ejecución de Tool: summarize_topic
  → Tool ejecutada exitosamente
  → 10 documentos FAISS trackeados para auditoria

[NODO] Evaluación
  → VÁLIDA (paso evaluación)

[NODO] Finalización (Intentos: 0)
  → Resultado final preparado

RESULTADO FINAL
Intención: resumen
Tool usada: summarize_topic
Chunks usados: 10
Válida: SÍ
Confianza: 90%

────────────────────────────────────────────────────────────────────────────────
RESPUESTA:
────────────────────────────────────────────────────────────────────────────────
TEMA: Resume el tema de Nitrile hydratases
SECCIONES: 5

  1. DEFINICIÓN
     Las nitrile hidratasas (NHases) son metaloenzimas que contienen un ion Fe(III) no hemo (tipo Fe) o un ion Co(III) no corrin (tipo Co) en su sitio activo. Catalizan

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PRUEBA 4: CONSULTA GENERAL (SIN RAG)
# ══════════════════════════════════════════════════════════════════════════════
# Prueba con una pregunta fuera del dominio (no sobre biología).
# El clasificador detecta intención "general" y ejecuta:
#   1. Sin recuperar documentos (no aplica RAG)
#   2. Responde directamente con Gemini sin contexto
#   3. Se marca como válida automáticamente
# 
# Resultado: Respuesta rápida sin procesamiento RAG.
# Demuestra que el sistema es flexible para múltiples dominios.

resultado = ejecutar_rag("¿Cuántos continentes hay?")
mostrar_resultado(resultado)


EJECUCIÓN DEL SISTEMA RAG

[NODO] Clasificación
  Consulta: ¿Cuántos continentes hay?
  → Intención: general
  → Top-K: 0

[NODO] Respuesta General (sin RAG)
  → Respuesta generada

[NODO] Finalización (Intentos: 0)
  → Resultado final preparado

RESULTADO FINAL
Intención: general
Tool usada: 
Chunks usados: 0
Válida: SÍ
Confianza: 100%

────────────────────────────────────────────────────────────────────────────────
RESPUESTA:
────────────────────────────────────────────────────────────────────────────────
Como experto en Biología Celular y Molecular, mi área de especialización se centra en el estudio de las células y los procesos moleculares que ocurren dentro de ellas. La pregunta sobre la cantidad de continentes pertenece al ámbito de la geografía.

Sin embargo, para responder a tu pregunta desde una perspectiva geográfica general, la respuesta más comúnmente aceptada es que hay **siete continentes**:

1.  **Asia**
2.  **África**
3.  **América del Norte**
4.  **América del Sur**
5